# Risk Horizon — run the pipeline

Runs every step of the final model in order. Set `ROOT` to the dataset folder (layout: `<ROOT>/2_readable/10_model_inputs/...`, see `05_model/README.md`). Crash-level inputs (`crash_events.csv`, `when_crash_hours.csv`) come from the cleaned iRAD release.

| Step | Script | Time |
|---|---|---|
| 0 | export model inputs to CSV | ~1 min |
| 1 | feature study (selection on 2023–mid 2024 → validation mid 2024–mid 2025) | ~15 min |
| 2 | static risk: train Jan 2023–Jun 2025, test once on Jul 2025–May 2026 | ~3 min |
| 3 | condition likelihood (case-crossover, conditional logit) | ~2 min |
| 4 | severity multipliers (darkness × lighting) | <1 min |
| 5 | signal bands 0–3 per 500 m + vehicle actions | <1 min |

In [ ]:
ROOT = r'path/to/RiskHorizon_Dataset'   # <- change me

## 0 · Readable model inputs
Converts model tables to CSV; from here on every model reads CSV only.

In [ ]:
!python ../04_model_inputs/code/00_export_model_inputs.py "{ROOT}"

## 1 · Feature study (run once)
Engineers 52 candidate features (`rh_features.py`), removes redundancy, tests PCA, measures each source by ablation, eliminates weak features, tunes the trees and picks the blend weight — using validation months only. Writes `final_features.json`.

In [ ]:
!python v2_final/01_feature_study.py "{ROOT}"

## 2 · WHERE: static risk
Negative-binomial safety performance function + boosted Poisson trees → blend 0.2/0.8 → Empirical Bayes. Prospective test with bootstrap confidence intervals.

In [ ]:
!python v2_final/02_static_risk.py "{ROOT}"

## 3 · WHEN: does the condition make a crash more likely?

In [ ]:
!python v2_final/03_condition_likelihood.py "{ROOT}"

## 4 · WHEN: does the condition make a crash deadlier?

In [ ]:
!python v2_final/04_severity_multipliers.py "{ROOT}"

## 5 · The signal: bands 0–3 per 500 m

In [ ]:
!python v2_final/05_signal_bands.py "{ROOT}"

## Results
See `06_results/README.md`.